## Benchmark with other methods

Categories:
- (Multi-omics) latent representation
- Gradient computation & Trajectory inference

In [ ]:
import os
import gc
import sys

import numpy as np
import cupy as cp
import pandas as pd
import scanpy as sc
import squidpy as sq

import torch
import torch.nn as nn
import tifffile
import seaborn as sns
import matplotlib.pyplot as plt

from torch_geometric import utils as pyg_utils
from scipy.stats import spearmanr
from sklearn.metrics import normalized_mutual_info_score

In [ ]:
from ipywidgets import interact, widgets
from IPython.display import display

from matplotlib import rcParams
rcParams['font.family'] = 'FreeSans'
rcParams.update({'font.size': 10})
rcParams.update({'figure.dpi': 300})
rcParams.update({'savefig.dpi': 300})

import warnings
warnings.filterwarnings('ignore')

%matplotlib inline

In [ ]:
sys.path.append('..')
sys.path.append('../models/')
sys.path.append('../util')

import IO, utils, trajectory, configs, gen_graph, zonation
import vgae, model_train, model_eval
import scFates as scf

In [ ]:
from importlib import reload
reload(trajectory)

In [ ]:
# Utils
from sklearn.metrics import roc_auc_score, f1_score

def compute_spearman(zonation, adata_antibody):
    factors = np.unique(zonation)
    corrs = np.zeros((len(factors), len(adata_antibody.var_names)))
    
    for i, factor in enumerate(factors):
        for j, label in enumerate(adata_antibody.var_names):
            corrs[i, j] = spearmanr(
                (zonation == factor).astype(np.uint8).values,
                adata_antibody[:, label].X.squeeze()   
            )[0]
            
    return corrs


def compute_auroc(t, adata_antibody, threshold=0.5):
    auroc = np.zeros_like(adata_antibody.var_names)
    for i, marker in enumerate(adata_antibody.var_names):
        target = adata_antibody[:, marker].X.squeeze() > threshold
        if 'GS' in marker or 'CYP' in marker:
            try:
                auroc[i] = roc_auc_score(target, t)
            except ValueError:
                auroc[i] = 0
        else:
            try:
                auroc[i] = roc_auc_score(target, 1-t)
            except ValueError:
                auroc[i] = 0

    return auroc


def kendall_tau_distance(u, v):
    """
    Compute normalized Kendall Tau distance btw 2 (permutated) rankings
    Reference: https://en.wikipedia.org/wiki/Kendall_tau_distance
    """
    n = len(u)
    assert len(v) == n, "Both lists have to be of equal length"
    i, j = np.meshgrid(np.arange(n), np.arange(n))
    a = np.argsort(u)
    b = np.argsort(v)
    ndisordered = np.logical_or(np.logical_and(a[i] < a[j], b[i] > b[j]), np.logical_and(a[i] > a[j], b[i] < b[j])).sum()
    return ndisordered / (n * (n - 1))


Load ample data: <br>

TODO: full training on all samples

In [ ]:
xenium_path = '../data/xenium/'
desi_path = '../data/integrated/desi_xenium_map/'
ab_path = '../data/integrated/antibody/'

sample_id = 'NIH_F5'
adata = IO.load_xenium(os.path.join(xenium_path, sample_id)) 
adata_desi = IO.load_desi(os.path.join(desi_path, sample_id+'.h5ad'), raw_img=False)
adata, adata_desi = IO.filter_cells(adata, adata_desi)

# Antibody (validation)
adata_ab = IO.load_ab_stain(
    os.path.join(ab_path, sample_id+'.ome.tif'),
    adata_ref=adata
)

ab_dict = {
    'Opal 690-GS': 'Central Vein',
    'Opal 780-CYP3A4': 'Peri-central',
    'Opal 570-ASS1': 'Peri-portal',
    'Opal 520-Col1': 'Portal Vein'
}
ab_labels = list(ab_dict.keys())

### scVI / MultiVI

In [ ]:
import scvi
scvi.settings.seed = 0
print("Last run with scvi-tools version:", scvi.__version__)

#### scVI

Model training:

In [ ]:
adata = adata.copy()
scvi.model.SCVI.setup_anndata(adata)

In [ ]:
n_latent = 8
model = scvi.model.SCVI(
    adata, 
    n_layers=2, 
    n_latent=n_latent, 
    gene_likelihood="nb"
)

model.train()

In [ ]:
latent = model.get_latent_representation()
np.save('../results/scvi_{}.npy'.format(n_latent), latent)
adata.obsm['X_scvi'] = latent

Trajectory inference:

In [ ]:
n_latent = 8
latent = np.load('../results/scvi_{}.npy'.format(n_latent))
adata.obsm['X_scvi'] = latent

In [ ]:
scf.tl.curve(
    adata,
    use_rep='X_scvi',
    Nodes=n_latent, 
    epg_extend_leaves=True,
    ndims_rep=n_latent
)

trajectory.compute_trajectory(
    adata, 
    use_rep='X_scvi',
    dist_metric='euclidean', 
    k=0
)

# Optional: rotate trajectory direction by 180-deg
adata.obs['t'] = adata.obs['t'].max() - adata.obs['t']
adata.obs['milestones'] = adata.obs['milestones'].astype(int).max() - adata.obs['milestones'].astype(int)
adata.obs['milestones'] = adata.obs['milestones'].astype('category')

t_scvi = adata.obs['t']
zone_scvi = adata.obs['milestones']

In [ ]:
# UMAP visualization
sc.pp.neighbors(adata, use_rep='X_scvi')
sc.tl.umap(adata)

# sns.heatmat(np.corrcoef(latent.T))
scf.pl.graph(adata, basis='umap', nodes=list(np.arange(n_latent))) 

In [ ]:
# Spatial visualization
sq.pl.spatial_scatter(
    adata, color='t', 
    vmin=np.quantile(adata.obs['t'], .05),
    vmax=np.quantile(adata.obs['t'], .95),
    cmap='RdBu_r', size=20, img=False,
    title='Pseudotime\n (scVI)'
)

sq.pl.spatial_scatter(
    adata, color='milestones', 
    cmap='tab10', size=20, img=False,
    title='Zonation\n (scVI)'
)

Antibody validation:

In [ ]:
# AUROC score against thresholded antibody channel
auroc_scvi = np.array([
    compute_auroc(t_scvi, adata_ab, threshold)
    for threshold in np.linspace(0.1, 0.9, 9)
])

Spatial heterogeneity metrics:

- (1). Moran's I, Geary's C (Ripley: measure cell density instead)
- (2). Network's centrality
  - degree centrality: % non-group members connected to group members
  - closeness centrality: closeness the group to other nodes

In [ ]:
# Spatial heterogeneity metrics
sq.gr.spatial_neighbors(adata, coord_type='generic', delaunay=True)

sq.gr.spatial_autocorr(adata, attr='obs', genes=['t'],
                       mode='moran', transformation=False)

sq.gr.spatial_autocorr(adata, attr='obs', genes=['t'],
                       mode='geary', transformation=False)
sq.gr.centrality_scores(adata, cluster_key='milestones')

display(adata.uns['moranI'])
display(adata.uns['gearyC'])
display(adata.uns['milestones_centrality_scores'])

moran_scvi = adata.uns['moranI'].loc['t', 'I']
geary_scvi = adata.uns['gearyC'].loc['t', 'C']
centrality_scvi = adata.uns['milestones_centrality_scores']['degree_centrality'].values


#### TotalVI / MultiVI

In [ ]:
import anndata as ad

In [ ]:
# Concatenate `var` for joint Xenium & DESI
n_genes = adata.shape[1]
n_metabolites = adata_desi.shape[1]

adata_mvi = ad.concat((adata, adata_desi), axis=1, join='inner')
adata_mvi.var['modality'] = ['Gene Expression']*n_genes + ['DESI Intensity']*n_metabolites
adata_mvi.obsm['spatial'] = adata.obsm['spatial'].copy()
adata_mvi.uns['spatial'] = adata.uns['spatial'].copy()

In [ ]:
scvi.model.MULTIVI.setup_anndata(adata_mvi)

In [ ]:
n_latent = 8
n_hidden = 16

model = scvi.model.MULTIVI(
    adata_mvi,
    n_genes=n_genes,
    n_regions=n_metabolites,
    n_hidden=n_hidden,
    n_latent=n_latent,
    gene_likelihood="nb"
)

In [ ]:
model.train(adversarial_mixing=False)

In [ ]:
latent = model.get_latent_representation()
adata.obsm['X_multivi'] = latent
np.save('../results/multivi_{}.npy'.format(n_latent), latent)

Trajectory inference:

In [ ]:
n_latent = 8
latent = np.load('../results/multivi_{}.npy'.format(n_latent))
adata.obsm['X_multivi'] = latent

In [ ]:
scf.tl.curve(adata,
             use_rep='X_multivi',
             Nodes=n_latent, 
             epg_extend_leaves=True,
             ndims_rep=n_latent)

trajectory.compute_trajectory(adata, 
                              use_rep='X_multivi',
                              dist_metric='euclidean', 
                              k=0)

# Optional: rotate trajectory direction by 180-deg
adata.obs['t'] = adata.obs['t'].max() - adata.obs['t']
adata.obs['milestones'] = adata.obs['milestones'].astype(int).max() - adata.obs['milestones'].astype(int)
adata.obs['milestones'] = adata.obs['milestones'].astype('category')

t_multivi = adata.obs['t']
zone_multivi = adata.obs['milestones']

In [ ]:
# UMAP visualization
sc.pp.neighbors(adata, use_rep='X_multivi')
sc.tl.umap(adata)
scf.pl.graph(adata, basis='umap', nodes=list(np.arange(n_latent))) 

In [ ]:
# Spatial visualization
sq.pl.spatial_scatter(
    adata, color='t', 
    vmin=np.quantile(adata.obs['t'], .05),
    vmax=np.quantile(adata.obs['t'], .95),
    cmap='RdBu_r', size=20, img=False,
    title='Pseudotime\n (MultiVI)'
)

sq.pl.spatial_scatter(
    adata, color='milestones', 
    cmap='tab10', size=20, img=False,
    title='Zonation\n (MultiVI)'
)

MultiVI:

In [ ]:
# AUROC score against thresholded antibody channel
auroc_multivi = np.array([
    compute_auroc(t_multivi, adata_ab, threshold)
    for threshold in np.linspace(0.1, 0.9, 9)
])

In [ ]:
# Spatial heterogeneity metrics
# sq.gr.spatial_neighbors(adata, coord_type='generic', delaunay=True)

sq.gr.spatial_autocorr(adata, attr='obs', genes=['t'],
                       mode='moran', transformation=False)

sq.gr.spatial_autocorr(adata, attr='obs', genes=['t'],
                       mode='geary', transformation=False)
sq.gr.centrality_scores(adata, cluster_key='milestones')

display(adata.uns['moranI'])
display(adata.uns['gearyC'])
display(adata.uns['milestones_centrality_scores'])

moran_multivi = adata.uns['moranI'].loc['t', 'I']
geary_multivi = adata.uns['gearyC'].loc['t', 'C']
centrality_multivi = adata.uns['milestones_centrality_scores']['degree_centrality'].values


**TODO: KDE(Raw Antibody expressions per zone)**

### CLUE

In [ ]:
import anndata as ad
import scglue
import networkx as nx

In [ ]:
# Configure model
scglue.models.configure_dataset(
    adata, "NB", use_highly_variable=False
)

scglue.models.configure_dataset(
    adata_desi, "Normal", use_highly_variable=False
)

In [ ]:
# First try without pretraining
n_latent = 8
n_hidden = 16

model = scglue.models.SCCLUEModel(
    adatas={"rna": adata, "metabolite": adata_desi},
    latent_dim=n_latent,
    x2u_h_dim=n_hidden,
    u2x_h_dim=n_hidden,
    random_seed=42
)

model.compile()
model.fit(adatas={"rna": adata, "metabolite": adata_desi})

In [ ]:
# DEBUG: why `n_latent` doubled?? multi-modal concatenation within `model.encode_data`? 
adata.obsm['X_glue'] = model.encode_data("rna", adata)
adata_desi.obsm['X_glue'] = model.encode_data("metabolite", adata_desi)
np.save('../results/clue_{}.npy'.format(n_latent), adata.obsm['X_glue'])

Trajectory inference:

In [ ]:
n_latent = 8 
latent = np.load('../results/clue_{}.npy'.format(n_latent))
adata.obsm['X_glue'] = latent

In [ ]:
# Debug: why `n_latent` doubled?: Confirm w/ author: concatenation of q(z1 | x1) || q(z1 | x2)

n_latent = 16  # tmp
n_nodes = 8
scf.tl.curve(
    adata,
    use_rep='X_glue',
    Nodes=n_nodes, 
    epg_extend_leaves=True,
    ndims_rep=n_latent
)

trajectory.compute_trajectory(
    adata, 
    use_rep='X_glue',
    dist_metric='euclidean', 
    k=0
)
n_latent = 8

# Optional: rotate trajectory direction by 180-deg
adata.obs['t'] = adata.obs['t'].max() - adata.obs['t']
adata.obs['milestones'] = adata.obs['milestones'].astype(int).max() - adata.obs['milestones'].astype(int)
adata.obs['milestones'] = adata.obs['milestones'].astype('category')

t_clue = adata.obs['t']
zone_clue = adata.obs['milestones']

In [ ]:
# UMAP visualization
sc.pp.neighbors(adata, use_rep='X_glue')
sc.tl.umap(adata)
scf.pl.graph(adata, basis='umap', nodes=list(np.arange(n_nodes))) 

In [ ]:
# Spatial visualization
sq.pl.spatial_scatter(
    adata, color='t', 
    vmin=np.quantile(adata.obs['t'], .05),
    vmax=np.quantile(adata.obs['t'], .95),
    cmap='RdBu_r', size=20, img=False,
    title='Pseudotime\n (CLUE)'
)

sq.pl.spatial_scatter(
    adata, color='milestones', 
    cmap='tab10', size=20, img=False,
    title='Zonation\n (CLUE)'
)

Antibody validation:

In [ ]:
# AUROC score against thresholded antibody channel
auroc_clue = np.array([
    compute_auroc(t_clue, adata_ab, threshold)
    for threshold in np.linspace(0.1, 0.9, 9)
])

In [ ]:
# Spatial heterogeneity metrics
# sq.gr.spatial_neighbors(adata, coord_type='generic', delaunay=True)

sq.gr.spatial_autocorr(adata, attr='obs', genes=['t'],
                       mode='moran', transformation=False)

sq.gr.spatial_autocorr(adata, attr='obs', genes=['t'],
                       mode='geary', transformation=False)
sq.gr.centrality_scores(adata, cluster_key='milestones')

display(adata.uns['moranI'])
display(adata.uns['gearyC'])
display(adata.uns['milestones_centrality_scores'])

moran_clue = adata.uns['moranI'].loc['t', 'I']
geary_clue = adata.uns['gearyC'].loc['t', 'C']
centrality_clue = adata.uns['milestones_centrality_scores']['degree_centrality'].values


### MOFA+

In [ ]:
import muon as mu

In [ ]:
adata.var['highly_variable'] = True
adata_desi.var['highly_variable'] = True

mdata = mu.MuData({"rna": adata, "metabolite": adata_desi})
mdata.var['highly_variable']=mdata.var['highly_variable'].to_list()
mdata

In [ ]:
n_latent = 8
mu.tl.mofa(mdata, n_factors=n_latent, seed=42)

In [ ]:
np.save('../results/mofa_{}.npy'.format(n_latent), mdata.obsm['X_mofa'])

Trajectory Inference:

In [ ]:
n_latent = 8 
latent = np.load('../results/mofa_{}.npy'.format(n_latent))
adata.obsm['X_mofa'] = latent

In [ ]:
scf.tl.curve(
    adata,
    use_rep='X_mofa',
    Nodes=n_latent, 
    epg_extend_leaves=True,
    ndims_rep=n_latent
)

trajectory.compute_trajectory(
    adata, 
    use_rep='X_mofa',
    dist_metric='euclidean', 
    k=0
)

# Optional: rotate trajectory direction by 180-deg
adata.obs['t'] = adata.obs['t'].max() - adata.obs['t']
adata.obs['milestones'] = adata.obs['milestones'].astype(int).max() - adata.obs['milestones'].astype(int)
adata.obs['milestones'] = adata.obs['milestones'].astype('category')

t_mofa = adata.obs['t']
zone_mofa = adata.obs['milestones']

In [ ]:
# UMAP visualization
sc.pp.neighbors(adata, use_rep='X_mofa')
sc.tl.umap(adata)
scf.pl.graph(adata, basis='umap', nodes=list(np.arange(n_latent))) 

In [ ]:
# Spatial visualization
sq.pl.spatial_scatter(
    adata, color='t', 
    vmin=np.quantile(adata.obs['t'], .03),
    vmax=np.quantile(adata.obs['t'], .97),
    cmap='RdBu_r', size=20, img=False,
    colorbar=None,
    title='Zonation trajectory\n (MOFA+)',
    # save='../results/plot/mofa.pdf'
)

# sq.pl.spatial_scatter(
#     adata, color='milestones', 
#     cmap='tab10', size=20, img=False,
#     title='Zonation\n (MOFA+)'
# )

In [ ]:
# AUROC score against thresholded antibody channel
auroc_mofa = np.array([
    compute_auroc(t_mofa, adata_ab, threshold)
    for threshold in np.linspace(0.1, 0.9, 9)
])

In [ ]:
# Spatial heterogeneity metrics
# sq.gr.spatial_neighbors(adata, coord_type='generic', delaunay=True)

sq.gr.spatial_autocorr(adata, attr='obs', genes=['t'],
                       mode='moran', transformation=False)

sq.gr.spatial_autocorr(adata, attr='obs', genes=['t'],
                       mode='geary', transformation=False)
sq.gr.centrality_scores(adata, cluster_key='milestones')

display(adata.uns['moranI'])
display(adata.uns['gearyC'])
display(adata.uns['milestones_centrality_scores'])

moran_mofa = adata.uns['moranI'].loc['t', 'I']
geary_mofa = adata.uns['gearyC'].loc['t', 'C']
centrality_mofa = adata.uns['milestones_centrality_scores']['degree_centrality'].values


### LYNX

In [ ]:
# Load vMF model & paired modalities
n_aux = 10
prior_choice = 'vMF'

# Compute auxiliary PCA for `u`
utils.get_PCs(adata_desi, n_pcs=n_aux)
adata.obsm['X_aux'] = adata_desi.obsm['X_pca'].astype(np.float32)

In [ ]:
# Load latent representation 
latent = utils.pnorm(np.load('../results/vMF_8.npy'))
adata.obsm['X_z'] = latent

Trajectory inference:

In [ ]:
scf.tl.curve(
    adata,
    use_rep='X_z',
    Nodes=n_latent, 
    epg_extend_leaves=True,
    ndims_rep=n_latent
)

trajectory.compute_trajectory(
    adata, 
    use_rep='X_z',
    dist_metric=prior_choice, 
    k=0
)

# Optional: rotate trajectory direction by 180-deg
adata.obs['t'] = adata.obs['t'].max() - adata.obs['t']
adata.obs['milestones'] = adata.obs['milestones'].astype(int).max() - adata.obs['milestones'].astype(int)
adata.obs['milestones'] = adata.obs['milestones'].astype('category')

t_svgae = adata.obs['t']
zone_svgae = adata.obs['milestones']

In [ ]:
# UMAP visualization
sc.pp.neighbors(adata, use_rep='X_z')
sc.tl.umap(adata)
scf.pl.graph(adata, basis='umap', nodes=list(np.arange(n_latent))) 

In [ ]:
# Spatial visualization
sq.pl.spatial_scatter(
    adata, color='t', 
    vmin=np.quantile(adata.obs['t'], .03),
    vmax=np.quantile(adata.obs['t'], .97),
    cmap='RdBu_r', size=20, img=False,
    colorbar=None,
    title='Zonation trajectory\n'+'LYNX',
    save='../results/plot/lynx.pdf'
)

# sq.pl.spatial_scatter(
#     adata, color='milestones', 
#     cmap='tab10', size=20, img=False,
#     title='Zonation\n'+'LYNX'
# )

In [ ]:
# AUROC score against thresholded antibody channel
auroc_svgae = np.array([
    compute_auroc(t_svgae, adata_ab, threshold)
    for threshold in np.linspace(0.1, 0.9, 9)
])

In [ ]:
# Spatial heterogeneity metrics
# sq.gr.spatial_neighbors(adata, coord_type='generic', delaunay=True)

sq.gr.spatial_autocorr(adata, attr='obs', genes=['t'],
                       mode='moran', transformation=False)

sq.gr.spatial_autocorr(adata, attr='obs', genes=['t'],
                       mode='geary', transformation=False)
sq.gr.centrality_scores(adata, cluster_key='milestones')

display(adata.uns['moranI'])
display(adata.uns['gearyC'])
display(adata.uns['milestones_centrality_scores'])

moran_svgae = adata.uns['moranI'].loc['t', 'I']
geary_svgae = adata.uns['gearyC'].loc['t', 'C']
centrality_svgae = adata.uns['milestones_centrality_scores']['degree_centrality'].values


### Metrics summary

In [ ]:
ts = [t_scvi, t_multivi, t_clue, t_mofa, t_svgae]
zones = [zone_scvi, zone_multivi, zone_clue, zone_mofa, zone_svgae]
rocs = np.array([auroc_scvi, auroc_multivi, auroc_clue, auroc_mofa, auroc_svgae])
methods = ['scVI', 'MultiVI', 'CLUE', 'MOFA+', 'LYNX']

In [ ]:
# AUROC score against thresholded antibodies

thresholds = np.linspace(0.1, 0.9, 9)
n_thresholds = auroc_scvi.shape[0]
n_antibodies = auroc_scvi.shape[1]
n_methods = len(methods)
n_repeats = auroc_scvi.shape[0]*auroc_scvi.shape[1]

plot_df = pd.DataFrame({
    'AUROC': rocs.flatten(),
    'Methods': np.repeat(methods, n_repeats),
    'threshold': np.tile(np.repeat(thresholds, n_antibodies), n_methods).flatten()
})


fig, ax = plt.subplots(figsize=(5, 3.5), dpi=300)
sns.lineplot(data=plot_df, x='threshold', y='AUROC', hue='Methods', 
             marker='o', markersize=5, legend=False,
             errorbar='se', ax=ax)
ax.set_title('AUROC vs. Antibody Validation', y=1.05, fontsize=12)
ax.spines[['right', 'top']].set_visible(False)
# ax.legend(loc='right', fontsize=8, bbox_to_anchor=(1.0, 0.9))
ax.set_xlabel('Antibody threshold')
ax.set_ylim([0, 1])

indices = np.random.choice(np.arange(n_thresholds-1), n_methods, replace=False)
for i, line, label in zip(indices, ax.lines, plot_df['Methods'].unique()):
    x_data = line.get_xdata()
    y_data = line.get_ydata()

    rotn = np.degrees(np.arctan2(
        y_data[i+1]-y_data[i],
        x_data[i+1]-x_data[i]
    ))
    ax.text(
        x_data[i]+.05, y_data[i]-.01, label,
        horizontalalignment='left', size='x-small', weight='bold',
        ha='center', va='bottom',
        rotation=rotn, rotation_mode='anchor', transform_rotates_text=True,
        color=line.get_color(),  
        bbox={'fc': 'lightgray', 'edgecolor': 'lightgray', 'pad': 0}
    )

plt.show()

del thresholds, n_thresholds, n_antibodies, n_methods, n_repeats
del x_data, y_data, indices

In [ ]:
fig.savefig('../results/plots/auroc.pdf', bbox_inches = "tight", dpi=300)

In [ ]:
# Fitted Antibody expressions in zones
import statsmodels.api as sm
from scipy.stats import zscore, gaussian_kde
from sklearn.preprocessing import QuantileTransformer

def disp_antibody_kdes(
    zonation, 
    adata_antibody,
    ncols=4,
    title=None,
    show_plot=True,
    return_density=False
):
    """
    Display KDE estimates of antibody distributions
    in discretized zonation assignments
    """
    markers = adata_antibody.var_names
    factors = np.unique(zonation)

    qt = QuantileTransformer()
    adata_ab = adata_antibody.copy()
    for marker in markers:
        expr = adata_antibody[:,  marker].X
        adata_ab[:, marker].X = qt.fit_transform(
            expr if isinstance(expr, np.ndarray) else
            expr.A
        )

    nrows = len(factors) // ncols
    if len(factors) % ncols != 0:
        nrows += 1

    densities = {}
    
    idx = 0
    fig, axes = plt.subplots(nrows, ncols, figsize=(5*ncols, 3.2*nrows))
    for r in range(nrows):
        for c in range(ncols):
            if idx >= len(factors):
                axes[r, c].axis('off')
                continue
                
            for marker in markers:                
                xx = adata_ab[zonation == factors[idx],  marker].X.squeeze()
                sns.kdeplot(xx, alpha=0.2, fill=True, ax=axes[r, c], label=marker)

                kde = sm.nonparametric.KDEUnivariate(xx).fit()
                densities.setdefault(marker, []).append((kde.support, kde.density))
                
            axes[r, c].set_title('Factor {}'.format(factors[idx]), fontsize=20)
            axes[r, c].set_xlabel('Expression', fontsize=15)
            axes[r, c].set_ylabel('Density', fontsize=15)
            axes[r, c].spines[['right', 'top']].set_visible(False)

            if idx == len(factors)-1:
                axes[r, c].legend(loc='right', bbox_to_anchor=(1.3, 0.5))
            idx += 1

    plt.tight_layout()
    plt.suptitle(title, y=1.05, fontsize=35)
    plt.show()

    if return_density:
        return densities
    return None


def get_antibody_distances(densities):
    """
    Compute pairwise distance of fitted antibody KDE peaks 
    along the discretized zonations
    """

    def _pairwise_wasserstain(values):
        values = np.array(values)
        distance = []
        for i in range(values.shape[0]-1):
            for j in range(i+1, values.shape[0]):
                distance.append(wasserstein_distance(values[i], values[j]))
        return distance
    
    antibody_labels = list(densities.keys())
    n_zones = len(densities[antibody_labels[0]])

    distances = []
    for i in range(n_zones):
        zone_densities = np.array([
            densities[label][i][1]
            for label in antibody_labels
        ])
        distances.append(_pairwise_wasserstain(zone_densities))
    return distances


In [ ]:
kde_distances = []
for zone, method in zip(zones, methods):
    densities = disp_antibody_kdes(zone, adata_ab, title=method, return_density=True)
    kde_distance = get_antibody_distances(densities)
    kde_distances.append(kde_distance)
del zone, method, kde_distance

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
labels = methods
counts = [moran_scvi, moran_multivi, moran_clue, moran_mofa, moran_svgae]
colors = ['tab:red', 'tab:blue', 'tab:green', 'tab:cyan', 'tab:orange']

ax.bar(labels, counts, label=labels, color=colors)
ax.set_xlabel('Method')
ax.set_ylabel(r"$I$")
ax.text(2, -.2, r"$higher\ is\ better$", ha='center', fontsize=8)
ax.set_title("Moran's I", fontsize=15)
ax.spines[['right', 'top']].set_visible(False)
plt.show()

del labels, counts, colors

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))

labels = methods
counts = [geary_scvi, geary_multivi, geary_clue, geary_mofa, geary_svgae]
colors = ['tab:red', 'tab:blue', 'tab:green', 'tab:cyan', 'tab:orange']

ax.bar(labels, counts, label=labels, color=colors)
ax.set_xlabel('Method')
ax.set_ylabel(r"$C$")
ax.set_title("Geary's C", fontsize=15)
ax.text(2, -.14, r"$lower\ is\ better$", ha='center', fontsize=8)
ax.spines[['right', 'top']].set_visible(False)
plt.show()

del labels, counts, colors

In [ ]:
# Centrality
n_nodes = len(centrality_scvi)
plot_df = pd.DataFrame({
    'Degree': list(centrality_scvi) + list(centrality_multivi) + list(centrality_clue) + \
              list(centrality_mofa) + list(centrality_svgae),
    'Method': ['scVI']*n_nodes + ['MultiVI']*n_nodes + ['CLUE']*n_nodes + \
              ['MOFA+']*n_nodes + ['LYNX']*n_nodes
})

fig, ax = plt.subplots(figsize=(6, 4))
ax = sns.boxplot(data=plot_df, x='Method', y='Degree', hue='Method', 
                 #errorbar=("ci", 95), 
                 #err_kws={"color": ".25", "linewidth": 1},
                 ax=ax)

ax.set_title('Degree of Centrality', fontsize=15)
ax.text(2, -.16, r"$lower\ is\ better$", ha='center', fontsize=8)
ax.spines[['right', 'top']].set_visible(False)
plt.show()

del n_nodes, plot_df

TODO: convert to table

In [ ]:
print([geary_scvi, geary_multivi, geary_clue, geary_mofa, geary_svgae])

In [ ]:
print([centrality_scvi.mean(), centrality_multivi.mean(), centrality_clue.mean(), centrality_mofa.mean(), centrality_svgae.mean()])
print([centrality_scvi.std(), centrality_multivi.std(), centrality_clue.std(), centrality_mofa.std(), centrality_svgae.std()])

In [ ]:
# Compute Pairwise Wdistances
for i in range(len(kde_distances)):
    print(methods[i], np.sum(kde_distances[i]))

---